# DemandForge — User Guide

**DemandForge** projects industrial hydrogen demand for EU-27 countries
across six sectors: **steel**, **refinery**, **ammonia**, **maritime**,
**olefins**, and **eSAF** (electro-Sustainable Aviation Fuel).

This notebook demonstrates:

1. Listing and inspecting scenario bundles
2. Running a single bundle with `load_bundle()`
3. Filtering by country and sector
4. Overriding individual parameters
5. Comparing scenarios
6. Accessing per-sector projection functions directly
7. Working with physical constants
8. Rich, publication-quality visualisations

In [ ]:
from __future__ import annotations
import sys, types, pathlib, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Sandbox compatibility stubs ────────────────────────────────────────────────────────────
if "platformdirs" not in sys.modules:
    _pd = types.ModuleType("platformdirs")
    _pd.user_data_dir = lambda *a, **kw: f"/tmp/{a[0] if a else 'data'}"
    _pd.user_cache_dir = lambda *a, **kw: f"/tmp/{a[0] if a else 'data'}"
    sys.modules["platformdirs"] = _pd
if "pyarrow" not in sys.modules:
    _pa = types.ModuleType("pyarrow"); _pa.__version__ = "0.0.0"
    _pac = types.ModuleType("pyarrow.compute")
    _pa.compute = _pac
    sys.modules["pyarrow"] = _pa
    sys.modules["pyarrow.compute"] = _pac
if "dotenv" not in sys.modules:
    _de = types.ModuleType("dotenv"); _de.load_dotenv = lambda *a, **kw: None
    sys.modules["dotenv"] = _de
_elec = types.ModuleType("demandforge.load_projection.electricity")
_elec.project_load_curve = lambda *a, **kw: None
sys.modules["demandforge.load_projection.electricity"] = _elec

# ── Path setup ───────────────────────────────────────────────────────────────────────────
try:
    BASE = pathlib.Path(__file__).resolve().parent.parent
except NameError:
    _cwd = pathlib.Path.cwd()
    BASE = _cwd if (_cwd / "demandforge").is_dir() else _cwd.parent
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch
import logging

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("demandforge")
logger.setLevel(logging.WARNING)

OUT_DIR = BASE / "notebooks"

# ── Colour palette ────────────────────────────────────────────────────────────────────────────
SECTOR_COLOURS = {
    "steel":    "#e63946",
    "refinery": "#457b9d",
    "ammonia":  "#2a9d8f",
    "maritime": "#264653",
    "olefins":  "#e9c46a",
    "esaf":     "#f4a261",
}
SCENARIO_COLOURS = {
    "central":         "#457b9d",
    "high_h2":         "#e63946",
    "low_h2":          "#2a9d8f",
    "industry_stress": "#e9c46a",
}
PATHWAY_COLOURS = {
    "mto":             "#e63946",
    "bio_naphtha":     "#2a9d8f",
    "chem_recycling":  "#457b9d",
    "fossil":          "#a8a8a8",
}

def _style_ax(ax, title="", ylabel="", xlabel="Year"):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", alpha=0.3, lw=0.5)


---
## 1 — Listing available scenario bundles

DemandForge ships four pre-defined bundles in `scenario_registry.yaml`.
Each bundles a coherent set of assumptions for all six sectors.

In [ ]:
from demandforge.load_projection.scenarios import list_bundles, get_bundle_params, load_bundle

bundles = list_bundles()
print("Available bundles:")
for name, desc in bundles.items():
    print(f"  • {name:20s}  {desc}")

---
## 2 — Inspecting bundle parameters

`get_bundle_params()` returns the raw YAML parameters without running
any projection — useful for auditing or programmatic modification.

In [ ]:
params = get_bundle_params("low_h2")
print("low_h2 scenario parameters:\n")
for sector, p in params.items():
    print(f"  {sector}:")
    if isinstance(p, dict):
        for k, v in p.items():
            if isinstance(v, dict):
                print(f"    {k}:")
                for k2, v2 in v.items():
                    if isinstance(v2, dict):
                        print(f"      {k2}: {v2}")
                    else:
                        print(f"      {k2}: {v2}")
            else:
                print(f"    {k}: {v}")


---
## 3 — Running a full scenario with `load_bundle()`

The simplest call: just a bundle name.  All six sectors run automatically
using package-default data (CONCAWE, WorldSteel, USGS, Eurostat).

In [ ]:
df = load_bundle("central")

print(f"Shape:     {df.shape}")
print(f"Columns:   {list(df.columns)}")
print(f"Sectors:   {sorted(df['sector'].unique())}")
print(f"Countries: {df['country'].nunique()}")
print(f"Years:     {df['year'].min()} – {df['year'].max()}")

df.head(10)

---
## 4 — Filtering by country

Pass a list of ISO-2 codes to restrict the computation.

In [ ]:
df_fr_de = load_bundle("central", countries=["FR", "DE"])
print(f"Countries: {sorted(df_fr_de['country'].unique())}")
print(f"Rows: {len(df_fr_de)}  (vs {len(df)} for all EU-27)")

---
## 5 — Filtering by sector

You can also run only a subset of sectors.

In [ ]:
df_steel_only = load_bundle("central", sectors=["steel"], countries=["DE"])
print(f"Sectors: {df_steel_only['sector'].unique()}")
print(f"DE steel H₂ at 2050: {df_steel_only[df_steel_only['year']==2050]['h2_demand_t_per_yr'].iloc[0]:,.0f} t/yr")

---
## 6 — Overriding parameters

Per-sector keyword arguments override the YAML bundle values.
The rest of the bundle stays intact.

In [ ]:
df_aggressive_dri = load_bundle(
    "central",
    countries=["DE"],
    sectors=["steel"],
    steel={"dri_share_2050": 1.0, "steel_cagr": 0.01},
)

df_base_dri = load_bundle("central", countries=["DE"], sectors=["steel"])

fig, ax = plt.subplots(figsize=(8, 4))
for label, d, ls in [
    ("central (DRI 60%)", df_base_dri, "-"),
    ("override (DRI 100%, +1% CAGR)", df_aggressive_dri, "--"),
]:
    agg = d.groupby("year")["h2_demand_mwh_per_yr"].sum() / 1e6
    ax.plot(agg.index, agg.values, ls=ls, lw=2.2, label=label)

_style_ax(ax, "DE Steel H₂ — parameter override demo", "TWh/yr")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_override_demo.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 7 — Comparing all four scenario bundles

The core use case: run every bundle and compare the demand trajectories.

In [ ]:
all_scenarios = {}
for bname in bundles:
    all_scenarios[bname] = load_bundle(bname)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for bname, colour in SCENARIO_COLOURS.items():
    df_b = all_scenarios[bname]
    total = df_b.groupby("year")["h2_demand_mwh_per_yr"].sum() / 1e6
    ax.plot(total.index, total.values, color=colour, lw=2.5, label=bname)
    ax.fill_between(total.index, 0, total.values, color=colour, alpha=0.08)

_style_ax(ax, "EU-27 total industrial H₂ demand — four scenarios", "TWh / yr")
ax.legend(fontsize=9, frameon=False, loc="upper left")
ax.set_xlim(2019, 2050)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.tight_layout()
fig.savefig(str(OUT_DIR / "EU-27 total industrial H₂ demand — four scenarios.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_c = all_scenarios["low_h2"]
pivot = df_c.pivot_table(
    index="year", columns="sector",
    values="h2_demand_mwh_per_yr", aggfunc="sum", fill_value=0,
) / 1e6  # TWh

ordered_sectors = ["steel", "refinery", "ammonia", "maritime", "olefins", "esaf"]
pivot = pivot[[s for s in ordered_sectors if s in pivot.columns]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.stackplot(
    pivot.index, *[pivot[s] for s in pivot.columns],
    labels=pivot.columns,
    colors=[SECTOR_COLOURS[s] for s in pivot.columns],
    alpha=0.85,
)
_style_ax(ax, "EU-27 H₂ demand by sector — low_h2 scenario", "TWh / yr")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], fontsize=8, frameon=False, loc="upper left")
ax.set_xlim(2019, 2050)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig2_sector_stacked.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_2050 = df_c[df_c["year"] == 2050]
top5 = (
    df_2050.groupby("country")["h2_demand_mwh_per_yr"]
    .sum().nlargest(5).index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 5))
bar_data = (
    df_2050[df_2050["country"].isin(top5)]
    .pivot_table(index="country", columns="sector",
                 values="h2_demand_mwh_per_yr", aggfunc="sum", fill_value=0)
    / 1e6
)
bar_data = bar_data.loc[top5][[s for s in ordered_sectors if s in bar_data.columns]]
bar_data.plot.bar(
    ax=ax, stacked=True, width=0.7,
    color=[SECTOR_COLOURS[s] for s in bar_data.columns],
    edgecolor="white", linewidth=0.5,
)
_style_ax(ax, "Top-5 countries at 2050 — sector breakdown (central)", "TWh / yr", "")
ax.legend(fontsize=8, frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
# Annotate totals
for i, cc in enumerate(top5):
    total = bar_data.loc[cc].sum()
    ax.text(i, total + 2, f"{total:.0f}", ha="center", fontsize=8, fontweight="bold")
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig3_top5_bar.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 8 — Per-sector evolution by country

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes_flat = axes.flatten()

for idx, sector in enumerate(ordered_sectors):
    ax = axes_flat[idx]
    df_sec = df_c[df_c["sector"] == sector]
    for cc in top5:
        ts = df_sec[df_sec["country"] == cc].set_index("year")["h2_demand_mwh_per_yr"] / 1e6
        ax.plot(ts.index, ts.values, lw=1.8, label=cc)
    _style_ax(ax, sector.upper(), "TWh/yr" if idx % 3 == 0 else "")
    ax.legend(fontsize=7, frameon=False, ncol=2)
    ax.set_xlim(2019, 2050)

fig.suptitle("H₂ demand evolution by sector — top 5 countries (central)", fontsize=13, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(str(OUT_DIR / "howto_fig4_sector_panels.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 9 — Scenario fan chart

Visualise the **uncertainty envelope** between low and high scenarios
for a single country.

In [ ]:
cc_focus = "FR"
fig, ax = plt.subplots(figsize=(10, 5))

# Compute total per scenario
series = {}
for bname in ["low_h2", "central", "high_h2", "industry_stress"]:
    df_b = all_scenarios[bname]
    ts = df_b[df_b["country"] == cc_focus].groupby("year")["h2_demand_mwh_per_yr"].sum() / 1e6
    series[bname] = ts

# Fill between low and stress
ax.fill_between(
    series["low_h2"].index,
    series["low_h2"].values,
    series["industry_stress"].values,
    alpha=0.12, color="#457b9d", label="low ↔ stress envelope",
)
ax.fill_between(
    series["low_h2"].index,
    series["low_h2"].values,
    series["high_h2"].values,
    alpha=0.15, color="#2a9d8f", label="low ↔ high envelope",
)

for bname, colour in SCENARIO_COLOURS.items():
    ax.plot(series[bname].index, series[bname].values, color=colour, lw=2.2, label=bname)

_style_ax(ax, f"{cc_focus} — total H₂ demand, scenario fan", "TWh / yr")
ax.legend(fontsize=8, frameon=False, loc="upper left")
ax.set_xlim(2019, 2050)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig5_fan_chart.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 10 — Sector share evolution (pie / donut)

In [ ]:
years_snap = [2030, 2050]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, yr in zip(axes, years_snap):
    df_yr = df_c[df_c["year"] == yr]
    shares = df_yr.groupby("sector")["h2_demand_mwh_per_yr"].sum()
    shares = shares.reindex(ordered_sectors).fillna(0)
    total_twh = shares.sum() / 1e6

    wedges, texts, autotexts = ax.pie(
        shares.values, labels=None,
        colors=[SECTOR_COLOURS[s] for s in shares.index],
        autopct=lambda pct: f"{pct:.0f}%" if pct > 3 else "",
        pctdistance=0.78, startangle=90,
        wedgeprops=dict(width=0.45, edgecolor="white", linewidth=1.5),
    )
    for at in autotexts:
        at.set_fontsize(8)
        at.set_fontweight("bold")

    ax.set_title(f"{yr}  —  {total_twh:.0f} TWh/yr", fontsize=11, fontweight="bold", pad=10)

# Shared legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=SECTOR_COLOURS[s], label=s) for s in ordered_sectors]
fig.legend(handles=legend_handles, fontsize=9, frameon=False,
           loc="center", bbox_to_anchor=(0.5, 0.02), ncol=6)
fig.suptitle("EU-27 sector shares — central scenario", fontsize=13, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig(str(OUT_DIR / "howto_fig6_donut.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 11 — Country heatmap at 2050

In [ ]:
df_2050_all = df_c[df_c["year"] == 2050]
heat = (
    df_2050_all.pivot_table(
        index="country", columns="sector",
        values="h2_demand_mwh_per_yr", aggfunc="sum", fill_value=0,
    ) / 1e6
)
heat = heat[[s for s in ordered_sectors if s in heat.columns]]
# Sort by total descending
heat["_total"] = heat.sum(axis=1)
heat = heat.sort_values("_total", ascending=True).drop(columns="_total")
# Keep only countries with > 0.5 TWh total
heat = heat[heat.sum(axis=1) > 0.5]

fig, ax = plt.subplots(figsize=(10, max(6, len(heat) * 0.35)))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, fontsize=9, rotation=30, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=9)

# Annotate cells
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        val = heat.iloc[i, j]
        if val > 0.1:
            colour = "white" if val > heat.values.max() * 0.6 else "black"
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7, color=colour)

cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("TWh / yr", fontsize=9)
ax.set_title("H₂ demand at 2050 by country × sector (central)", fontsize=11, fontweight="bold", pad=10)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig7_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 12 — Using per-sector projection functions directly

For fine-grained control, call individual projection functions from
`demandforge.load_projection.hydrogen`.

In [ ]:
from demandforge.load_projection.hydrogen import (
    project_ammonia_h2_demand,
    project_steel_h2_demand,
    project_maritime_h2_demand,
    project_olefins_h2_demand,
)
from demandforge.process.refinery import get_naphtha_for_crackers, CONCAWE_MORE_MOLECULE
from demandforge.load_projection.constants import PETROCHEM_NAPHTHA_FRACTION

# Ammonia with custom parameters
df_nh3 = project_ammonia_h2_demand(
    country=["FR", "DE"],
    reference_year=2019, target_year=2050,
    h2_route_share_end=0.90, domestic_share_end=0.80,
    decarb_start=2025, decarb_end=2045,
)
print(f"Ammonia projection: {df_nh3.shape[0]} rows")
print(f"  FR at 2050: {df_nh3[(df_nh3['country']=='FR') & (df_nh3['year']==2050)]['h2_demand_network_t_per_yr'].iloc[0]:,.0f} t/yr")

# Steel with aggressive DRI
df_steel = project_steel_h2_demand(
    country="DE", dri_share_2050=1.0, h2_per_t_dri=0.054, steel_cagr=0.005,
)
print(f"\nSteel projection (DE, DRI=100%): {df_steel.shape[0]} rows")
print(f"  DE at 2050: {df_steel[df_steel['year']==2050]['h2_demand_for_steel_t_per_yr'].iloc[0]:,.0f} t/yr")

# Olefins — unconstrained (no naphtha coupling)
df_olef_unc = project_olefins_h2_demand(
    country=["FR", "DE"], reference_year=2019, target_year=2050,
    olefins_cagr=0.005,
    pathways={
        "mto":            {"share_2050": 0.30, "ramp_start": 2028, "ramp_end": 2042},
        "bio_naphtha":    {"share_2050": 0.15, "ramp_start": 2024, "ramp_end": 2035},
        "chem_recycling": {"share_2050": 0.15, "ramp_start": 2026, "ramp_end": 2036},
    },
)

# Olefins — with naphtha supply constraint (more-molecule scenario)
_years = np.arange(2019, 2051)
nap_mm = get_naphtha_for_crackers(CONCAWE_MORE_MOLECULE, _years,
                                   petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION)
df_olef_con = project_olefins_h2_demand(
    country=["FR", "DE"], reference_year=2019, target_year=2050,
    olefins_cagr=0.005,
    pathways={
        "mto":            {"share_2050": 0.30, "ramp_start": 2028, "ramp_end": 2042},
        "bio_naphtha":    {"share_2050": 0.15, "ramp_start": 2024, "ramp_end": 2035},
        "chem_recycling": {"share_2050": 0.15, "ramp_start": 2026, "ramp_end": 2036},
    },
    naphtha_supply_mt=nap_mm, cracker_yield=0.45,
)

print(f"\nOlefins FR@2050 — unconstrained vs naphtha-constrained:")
for label, df in [("Unconstrained", df_olef_unc), ("Constrained", df_olef_con)]:
    row = df[(df['country']=='FR') & (df['year']==2050)]
    print(f"  {label:15s}: H2={row['h2_demand_t_per_yr'].iloc[0]:>8,.0f} t/yr  "
          f"MTO={row['mto_share'].iloc[0]:.0%}  bio={row['bio_naphtha_share'].iloc[0]:.0%}  "
          f"rec={row['chem_recycling_share'].iloc[0]:.0%}  fos={row['fossil_share'].iloc[0]:.0%}")

### Olefins pathway decomposition — unconstrained vs naphtha-constrained

The left panel shows the **unconstrained** pathway mix (YAML targets only).
The right panel shows the **naphtha-constrained** mix where fossil share is
capped by available refinery naphtha from the CONCAWE scenario.  The excess
is redistributed proportionally to the green pathways.

In [ ]:
# --- Olefins pathway decomposition: unconstrained vs constrained ---
from demandforge.process.olefins import (
    build_olefins_pathway_mix, compute_olefins_h2_demand,
    apply_naphtha_supply_constraint,
)
from demandforge.process.refinery import get_naphtha_for_crackers, CONCAWE_MORE_MOLECULE
from demandforge.load_projection.constants import PETROCHEM_NAPHTHA_FRACTION

years = np.arange(2019, 2051)
mix_unc = build_olefins_pathway_mix(
    years,
    mto_share_2050=0.20, mto_ramp_start=2030, mto_ramp_end=2045,
    bio_naphtha_share_2050=0.15, bio_naphtha_ramp_start=2024, bio_naphtha_ramp_end=2035,
    chem_recycling_share_2050=0.10, chem_recycling_ramp_start=2027, chem_recycling_ramp_end=2038,
)

# Constrained mix (EU production ~ 22 Mt, constant for central)
eu_prod = np.full(len(years), 22.4e6)
nap_mm = get_naphtha_for_crackers(CONCAWE_MORE_MOLECULE, years,
                                   petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION)
mix_con = apply_naphtha_supply_constraint(
    mix_unc.copy(), eu_prod, nap_mm, cracker_yield=0.45,
)

pathway_labels = ["MTO", "Bio-naphtha", "Chem. recycling", "Fossil"]
pathway_cols   = ["mto_share", "bio_naphtha_share", "chem_recycling_share", "fossil_share"]
colours = [PATHWAY_COLOURS[k] for k in ["mto", "bio_naphtha", "chem_recycling", "fossil"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mix, title in [
    (axes[0], mix_unc, "Unconstrained (YAML targets)"),
    (axes[1], mix_con, "Naphtha-constrained (central / more-molecule)"),
]:
    ax.stackplot(
        mix["year"], *[mix[c] for c in pathway_cols],
        labels=pathway_labels, colors=colours, alpha=0.85,
    )
    ax.set_xlim(years[0], years[-1])
    ax.set_ylim(0, 1)
    _style_ax(ax, title, "Production share")
    ax.legend(loc="center left", fontsize=8, frameon=False)

fig.tight_layout()
fig.savefig(OUT_DIR / "olefins_pathway_unconstrained_vs_constrained.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Fossil share at 2050: {mix_unc['fossil_share'].iloc[-1]:.0%} (unconstrained) "
      f"-> {mix_con['fossil_share'].iloc[-1]:.0%} (constrained)")

---
## 13 — Physical constants

All shared physical constants live in `load_projection.constants`:

In [ ]:
from demandforge.load_projection.constants import (
    H2_LHV_MWH_PER_T,
    H2_T_PER_T_NH3,
    H2_T_PER_T_E_METHANOL,
    H2_T_PER_T_NH3_FUEL,
    H2_INTENSITY_T_PER_T_ESAF,
    REFINERY_INEFFICIENCY_SHARE,
    H2_T_PER_T_OLEFIN_MTO,
    H2_T_PER_T_OLEFIN_BIO_NAPHTHA,
    H2_T_PER_T_OLEFIN_CHEM_RECYCL,
    H2_T_PER_T_OLEFIN_FOSSIL_HT,
    PETROCHEM_NAPHTHA_FRACTION,
    STEAM_CRACKER_OLEFIN_YIELD,
)

const_table = pd.DataFrame({
    "Constant": [
        "H2 LHV", "H2/NH3 (Haber-Bosch)", "H2/e-methanol",
        "H2/NH3 (fuel-grade)", "H2 intensity eSAF", "Refinery inefficiency",
        "H2/olefin MTO", "H2/olefin bio-naphtha", "H2/olefin chem recycling",
        "H2/olefin fossil HT", "Petrochem naphtha fraction",
        "Steam cracker olefin yield",
    ],
    "Value": [
        H2_LHV_MWH_PER_T, H2_T_PER_T_NH3, H2_T_PER_T_E_METHANOL,
        H2_T_PER_T_NH3_FUEL, H2_INTENSITY_T_PER_T_ESAF, REFINERY_INEFFICIENCY_SHARE,
        H2_T_PER_T_OLEFIN_MTO, H2_T_PER_T_OLEFIN_BIO_NAPHTHA,
        H2_T_PER_T_OLEFIN_CHEM_RECYCL, H2_T_PER_T_OLEFIN_FOSSIL_HT,
        PETROCHEM_NAPHTHA_FRACTION, STEAM_CRACKER_OLEFIN_YIELD,
    ],
    "Unit": [
        "MWh/t", "t H2/t NH3", "t H2/t MeOH",
        "t H2/t NH3", "t H2/t eSAF", "dimensionless",
        "t H2/t olefin", "t H2/t olefin", "t H2/t olefin",
        "t H2/t olefin", "dimensionless", "t olefin/t naphtha",
    ],
})
print(const_table.to_string(index=False))

> **Methodological note -- e-SAF baseline scope.**
>
> The e-SAF projection derives fossil jet availability from the CONCAWE
> *Kero Hydrotreater* unit feed (~3 % of refinery throughput, ~17-18 Mt/yr
> at 2019).  This is broadly consistent with EU-27 refinery jet/kerosene
> **production** (~10-12 Mt/yr, Eurostat `nrg_bal_c`, ~9.9 Mtoe in 2023);
> the Kero HT figure is slightly higher because it includes intermediate
> streams being hydrotreated, not only final sellable product.
>
> The commonly cited 50-60 Mt/yr figure refers to **consumption** (fuel
> uplifted at EU airports), not domestic production.  The EU is a large net
> importer of jet fuel (~75-80 % of demand sourced externally).
>
> The model computes the eSAF gap against the *refinery-production* ceiling,
> not total consumption.  This is the correct framing for a supply-side H2
> model: as domestic refinery capacity declines, imported fossil jet could
> still fill part of the demand.  If policy mandates e-SAF on **all**
> consumption (including imports), H2 demand would scale to the full 50+ Mt
> baseline (~1 000+ TWh).

> **Methodological note -- CONCAWE capacity delay.**
>
> The CONCAWE Low Carbon Pathways (2020) assume near-term refinery capacity
> declines of ~5-8 %/yr before 2030, exceeding historical EU closure rates
> (~1-2 %/yr).  To produce a more realistic transition path, the central,
> high_h2, and industry_stress bundles apply a **10-year capacity delay**:
> at year *t*, each CONCAWE unit's capacity equals the original trajectory
> value at *t - 10*, clamped to the 2024 observed level.
>
> Effect: capacity stays flat until ~2034, then follows the original
> decline shape.  At 2050, capacity equals the original 2040 value
> (level factor 0.275 instead of 0.102 for more-molecule).  This cascades
> to eSAF (more fossil jet available, less synthetic fuel needed) and olefins
> (looser naphtha constraint, less forced green-pathway substitution).
>
> The low_h2 bundle uses no delay (`capacity_delay_years: 0`), consistent
> with its sufficiency narrative where aggressive electrification drives
> refinery exit by 2050.
>
> The parameter is set in `scenario_registry.yaml` under `refinery:` and
> applied by `process.refinery.apply_capacity_delay()` before any downstream
> computation (naphtha extraction, petrochem correction, H2 demand).

---
## 14 — Scenario sensitivity: spider chart

How does each sector's H₂ demand at 2050 vary across scenarios?

In [ ]:
from math import pi

sectors_for_radar = ordered_sectors
n_sectors = len(sectors_for_radar)
angles = [n * 2 * pi / n_sectors for n in range(n_sectors)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Compute 2050 values per scenario per sector, normalised to industry_stress
ref_vals = {}
for s in sectors_for_radar:
    df_s = all_scenarios["industry_stress"]
    ref_vals[s] = df_s[(df_s["sector"] == s) & (df_s["year"] == 2050)]["h2_demand_mwh_per_yr"].sum()

for bname, colour in SCENARIO_COLOURS.items():
    df_b = all_scenarios[bname]
    df_2050b = df_b[df_b["year"] == 2050]
    vals = []
    for s in sectors_for_radar:
        v = df_2050b[df_2050b["sector"] == s]["h2_demand_mwh_per_yr"].sum()
        vals.append(v / ref_vals[s] * 100 if ref_vals[s] > 0 else 0)
    vals += vals[:1]
    ax.plot(angles, vals, color=colour, lw=2, label=bname)
    ax.fill(angles, vals, color=colour, alpha=0.06)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([s.upper() for s in sectors_for_radar], fontsize=9)
ax.set_title("Sector sensitivity at 2050\n(% of industry_stress)", fontsize=11,
             fontweight="bold", pad=20)
ax.legend(fontsize=8, frameon=False, loc="lower right", bbox_to_anchor=(1.25, -0.05))
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig8_spider.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 15 — Waterfall: scenario delta decomposition

Which sectors contribute most to the difference between low_h2 and high_h2?

In [ ]:
df_low_50  = all_scenarios["low_h2"][all_scenarios["low_h2"]["year"] == 2050]
df_high_50 = all_scenarios["high_h2"][all_scenarios["high_h2"]["year"] == 2050]

low_by_sec  = df_low_50.groupby("sector")["h2_demand_mwh_per_yr"].sum() / 1e6
high_by_sec = df_high_50.groupby("sector")["h2_demand_mwh_per_yr"].sum() / 1e6
delta = (high_by_sec - low_by_sec).reindex(ordered_sectors).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
cumulative = 0
low_total = low_by_sec.sum()
# Starting bar
ax.barh(0, low_total, color="#a8dadc", edgecolor="white", height=0.5)
ax.text(low_total / 2, 0, f"low_h2\n{low_total:.0f}", ha="center", va="center", fontsize=9, fontweight="bold")

for i, sector in enumerate(ordered_sectors):
    d = delta[sector]
    ax.barh(i + 1, d, left=low_total + cumulative, height=0.5,
            color=SECTOR_COLOURS[sector], edgecolor="white")
    pos = low_total + cumulative + d / 2
    if abs(d) > 5:
        ax.text(pos, i + 1, f"+{d:.0f}", ha="center", va="center", fontsize=8, fontweight="bold")
    cumulative += d

# Ending bar
high_total = high_by_sec.sum()
ax.barh(len(ordered_sectors) + 1, high_total, color="#e63946", edgecolor="white", height=0.5, alpha=0.7)
ax.text(high_total / 2, len(ordered_sectors) + 1, f"high_h2\n{high_total:.0f}",
        ha="center", va="center", fontsize=9, fontweight="bold", color="white")

labels = ["low_h2 total"] + [s.upper() for s in ordered_sectors] + ["high_h2 total"]
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
_style_ax(ax, "Waterfall: low_h2 → high_h2 delta at 2050 (EU-27)", "TWh / yr", "")
ax.set_xlabel("TWh / yr", fontsize=9)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig9_waterfall.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 16 — Neighbour comparison

How do adjacent countries compare?  Useful for sanity-checking geographic
coherence of the projections.

In [ ]:
NEIGHBOURS = [
    ("DE", "FR"), ("DE", "PL"), ("IT", "ES"),
    ("NL", "BE"), ("SE", "FI"), ("AT", "CZ"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (cc_a, cc_b) in zip(axes.flatten(), NEIGHBOURS):
    for cc, offset in [(cc_a, -0.18), (cc_b, 0.18)]:
        df_cc = df_c[(df_c["country"] == cc) & (df_c["year"] == 2050)]
        by_sec = df_cc.set_index("sector")["h2_demand_mwh_per_yr"].reindex(ordered_sectors).fillna(0) / 1e6
        x = np.arange(len(ordered_sectors))
        bars = ax.bar(x + offset, by_sec.values, width=0.35, label=cc,
                      color=[SECTOR_COLOURS[s] for s in ordered_sectors],
                      alpha=0.9 if offset < 0 else 0.55,
                      edgecolor="white", linewidth=0.5)
    ax.set_xticks(range(len(ordered_sectors)))
    ax.set_xticklabels([s[:4] for s in ordered_sectors], fontsize=7, rotation=30)
    _style_ax(ax, f"{cc_a} vs {cc_b}", "TWh/yr" if ax.get_subplotspec().colspan.start == 0 else "", "")
    ax.legend(fontsize=8, frameon=False)

fig.suptitle("Neighbour comparison at 2050 — central scenario", fontsize=13, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(str(OUT_DIR / "howto_fig10_neighbours.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 17 — Growth rate analysis

Compute compound annual growth rates (CAGR) from 2025 to 2050 by sector.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bar_width = 0.18
x = np.arange(len(ordered_sectors))

for i, (bname, colour) in enumerate(SCENARIO_COLOURS.items()):
    cagrs = []
    for sector in ordered_sectors:
        df_b = all_scenarios[bname]
        df_sec = df_b[df_b["sector"] == sector]
        val_2025 = df_sec[df_sec["year"] == 2025]["h2_demand_mwh_per_yr"].sum()
        val_2050 = df_sec[df_sec["year"] == 2050]["h2_demand_mwh_per_yr"].sum()
        if val_2025 > 0 and val_2050 > 0:
            cagr = (val_2050 / val_2025) ** (1 / 25) - 1
        else:
            cagr = 0.0
        cagrs.append(cagr * 100)

    ax.bar(x + i * bar_width, cagrs, bar_width, color=colour, label=bname, edgecolor="white")

ax.set_xticks(x + 1.5 * bar_width)
ax.set_xticklabels([s.upper() for s in ordered_sectors], fontsize=9)
_style_ax(ax, "Compound annual growth rate 2025–2050 by sector", "CAGR (%)")
ax.axhline(0, color="black", lw=0.5)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "howto_fig11_cagr.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 18 — Cumulative H₂ demand 2025–2050

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)

for ax, (bname, colour) in zip(axes, SCENARIO_COLOURS.items()):
    df_b = all_scenarios[bname]
    df_b_post = df_b[df_b["year"] >= 2025]
    cum = df_b_post.pivot_table(
        index="year", columns="sector",
        values="h2_demand_mwh_per_yr", aggfunc="sum", fill_value=0,
    ).cumsum() / 1e6  # cumulative TWh

    cum = cum[[s for s in ordered_sectors if s in cum.columns]]
    ax.stackplot(
        cum.index, *[cum[s] for s in cum.columns],
        colors=[SECTOR_COLOURS[s] for s in cum.columns],
        alpha=0.85,
    )
    total_cum = cum.iloc[-1].sum()
    _style_ax(ax, f"{bname}\n{total_cum:,.0f} TWh total", "Cumulative TWh" if ax == axes[0] else "")
    ax.set_xlim(2025, 2050)

# Shared legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=SECTOR_COLOURS[s], label=s) for s in ordered_sectors]
fig.legend(handles=legend_handles, fontsize=8, frameon=False,
           loc="center", bbox_to_anchor=(0.5, 0.01), ncol=6)
fig.suptitle("Cumulative H₂ demand 2025–2050 by scenario", fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig(str(OUT_DIR / "howto_fig12_cumulative.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 19 — Naphtha supply constraint impact

The endogenous refinery-olefins coupling caps fossil olefin production against
available naphtha from the CONCAWE scenario.  This section visualises how the
constraint reshapes the olefins pathway mix and H2 demand across all four bundles.

In [ ]:
# --- Naphtha constraint: fossil share before / after, all bundles ---
from demandforge.process.refinery import (
    get_naphtha_for_crackers, CONCAWE_MORE_MOLECULE, CONCAWE_MAX_ELECTRON,
)
from demandforge.load_projection.constants import PETROCHEM_NAPHTHA_FRACTION

years = np.arange(2019, 2051)
nap_mm = get_naphtha_for_crackers(CONCAWE_MORE_MOLECULE, years,
                                   petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION)
nap_me = get_naphtha_for_crackers(CONCAWE_MAX_ELECTRON, years,
                                   petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION)

NAP_MAP = {
    "central": nap_mm, "high_h2": nap_mm,
    "low_h2": nap_me, "industry_stress": nap_mm,
}

from demandforge.load_projection.hydrogen import project_olefins_h2_demand
from demandforge.load_projection.scenarios import get_bundle_params

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, bname in zip(axes.flatten(), ["central", "high_h2", "low_h2", "industry_stress"]):
    params = get_bundle_params(bname)["olefins"]
    nap = NAP_MAP[bname]

    df_unc = project_olefins_h2_demand(country=None, reference_year=2019,
                                        target_year=2050, **params)
    df_con = project_olefins_h2_demand(country=None, reference_year=2019,
                                        target_year=2050,
                                        naphtha_supply_mt=nap, cracker_yield=0.45,
                                        **params)

    row_u = df_unc[df_unc["country"] == df_unc["country"].iloc[0]]
    row_c = df_con[df_con["country"] == df_con["country"].iloc[0]]

    ax.plot(row_u["year"], row_u["fossil_share"], ls="--", lw=2,
            color=PATHWAY_COLOURS["fossil"], label="Fossil (unconstrained)")
    ax.plot(row_c["year"], row_c["fossil_share"], ls="-", lw=2,
            color=PATHWAY_COLOURS["fossil"], label="Fossil (constrained)")
    ax.plot(row_c["year"], row_c["mto_share"], ls="-", lw=2,
            color=PATHWAY_COLOURS["mto"], label="MTO (constrained)")
    ax.fill_between(row_c["year"], 0,
                    row_c["bio_naphtha_share"] + row_c["chem_recycling_share"],
                    color=PATHWAY_COLOURS["bio_naphtha"], alpha=0.2,
                    label="Bio + recycling (con.)")
    _style_ax(ax, bname, "Share")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, frameon=False, loc="center right")

fig.suptitle("Olefins fossil share: YAML target vs naphtha-constrained",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "naphtha_constraint_all_bundles.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 20 — Per-scenario demand breakdown at 2050

Grouped bar chart comparing all six sectors across the four bundles at
the 2050 horizon.  This shows how the aggregate demand envelope shifts
with scenario assumptions (including naphtha-constrained olefins).

In [ ]:
# --- Grouped bar chart: sector demand at 2050 across all bundles ---
fig, ax = plt.subplots(figsize=(12, 6))

bar_width = 0.18
x = np.arange(len(ordered_sectors))

for i, (bname, colour) in enumerate(SCENARIO_COLOURS.items()):
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    vals = []
    for sec in ordered_sectors:
        sec_sum = df_50[df_50["sector"] == sec]["h2_demand_mwh_per_yr"].sum() / 1e6
        vals.append(sec_sum)
    ax.bar(x + i * bar_width, vals, bar_width, label=bname, color=colour, alpha=0.85)

ax.set_xticks(x + 1.5 * bar_width)
ax.set_xticklabels([s.capitalize() for s in ordered_sectors], fontsize=9)
_style_ax(ax, "EU-27 H2 demand at 2050 by sector and scenario", "TWh / yr")
ax.legend(fontsize=9, frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(OUT_DIR / "sector_demand_2050_grouped.png", dpi=150, bbox_inches="tight")
plt.show()

# Print numeric summary
print(f"{'Sector':<12} {'central':>10} {'high_h2':>10} {'low_h2':>10} {'stress':>10}  (TWh)")
print("-" * 58)
for sec in ordered_sectors:
    vals = []
    for bname in SCENARIO_COLOURS:
        df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
        vals.append(df_50[df_50["sector"] == sec]["h2_demand_mwh_per_yr"].sum() / 1e6)
    print(f"{sec:<12} " + "  ".join(f"{v:>10.1f}" for v in vals))

total_row = []
for bname in SCENARIO_COLOURS:
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    total_row.append(df_50["h2_demand_mwh_per_yr"].sum() / 1e6)
print("-" * 58)
print(f"{'TOTAL':<12} " + "  ".join(f"{v:>10.1f}" for v in total_row))

### 20b — Country-level sector demand at 2050

The same grouped-bar view as above, but broken out for the **top-8 countries**
by total H₂ demand. Each subplot shows sector-by-scenario demand for one country.

In [ ]:
# --- Country-level grouped bar: sector demand at 2050 for top-8 countries ---
# Identify top-8 countries by total demand in the central scenario
df_c50 = all_scenarios['central'][all_scenarios['central']['year'] == 2050]
top8 = (
    df_c50.groupby('country')['h2_demand_mwh_per_yr'].sum()
    .nlargest(8).index.tolist()
)

fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey=False)
axes_flat = axes.flatten()

bar_width = 0.18
x = np.arange(len(ordered_sectors))

for ax_idx, cc in enumerate(top8):
    ax = axes_flat[ax_idx]
    for i, (bname, colour) in enumerate(SCENARIO_COLOURS.items()):
        df_50 = all_scenarios[bname][
            (all_scenarios[bname]['year'] == 2050) &
            (all_scenarios[bname]['country'] == cc)
        ]
        vals = []
        for sec in ordered_sectors:
            v = df_50[df_50['sector'] == sec]['h2_demand_mwh_per_yr'].sum() / 1e6
            vals.append(v)
        ax.bar(x + i * bar_width, vals, bar_width,
               label=bname if ax_idx == 0 else '', color=colour, alpha=0.85)
    ax.set_xticks(x + 1.5 * bar_width)
    ax.set_xticklabels([s[:4].capitalize() for s in ordered_sectors],
                       fontsize=7, rotation=45, ha='right')
    _style_ax(ax, cc, 'TWh / yr' if ax_idx % 4 == 0 else '')

fig.legend(
    [plt.Rectangle((0, 0), 1, 1, fc=c, alpha=0.85)
     for c in SCENARIO_COLOURS.values()],
    list(SCENARIO_COLOURS.keys()),
    loc='upper center', ncol=4, fontsize=9, frameon=False,
    bbox_to_anchor=(0.5, 1.02),
)
fig.suptitle('H$_2$ demand at 2050 — sector × scenario (top-8 countries)',
             fontsize=13, fontweight='bold', y=1.06)
fig.tight_layout()
fig.savefig(OUT_DIR / 'country_sector_demand_2050_grouped.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Print numeric table for top-8
print(f"{'Country':<8} {'Scenario':<18} " + '  '.join(f'{s:>8}' for s in ordered_sectors) + '  TOTAL (TWh)')
print('-' * 100)
for cc in top8:
    for bname in SCENARIO_COLOURS:
        df_50 = all_scenarios[bname][
            (all_scenarios[bname]['year'] == 2050) &
            (all_scenarios[bname]['country'] == cc)
        ]
        vals = [df_50[df_50['sector'] == s]['h2_demand_mwh_per_yr'].sum() / 1e6
                for s in ordered_sectors]
        total = sum(vals)
        print(f'{cc:<8} {bname:<18} ' + '  '.join(f'{v:>8.1f}' for v in vals) + f'  {total:>8.1f}')
    print()

---
## 21 — Country-level demand distribution

How is hydrogen demand distributed across EU-27 countries?  This section
shows the top-10 countries by demand at 2050, sector-level breakdowns per
country, and a geographic Lorenz curve of demand concentration.

In [ ]:
# --- Top-10 countries at 2050 by total H2 demand, all 4 bundles ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (bname, colour) in zip(axes.flatten(), SCENARIO_COLOURS.items()):
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    country_totals = df_50.groupby("country")["h2_demand_mwh_per_yr"].sum().sort_values(ascending=True)
    top10 = country_totals.nlargest(10).sort_values()

    bars = ax.barh(top10.index, top10.values / 1e6, color=colour, alpha=0.8, height=0.6)
    for bar, val in zip(bars, top10.values / 1e6):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{val:.0f}", va="center", fontsize=7)
    _style_ax(ax, f"{bname} — Top 10 countries at 2050", "TWh / yr", xlabel="")
    ax.set_xlabel("TWh / yr", fontsize=9)

fig.suptitle("Country-level H2 demand at 2050", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "country_top10_all_bundles.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Stacked bar: sector breakdown for top-8 countries (central) ---
df_50 = all_scenarios["central"][all_scenarios["central"]["year"] == 2050]
top8 = df_50.groupby("country")["h2_demand_mwh_per_yr"].sum().nlargest(8).index.tolist()

pivot_c = df_50[df_50["country"].isin(top8)].pivot_table(
    index="country", columns="sector", values="h2_demand_mwh_per_yr",
    aggfunc="sum", fill_value=0,
) / 1e6
pivot_c = pivot_c[[s for s in ordered_sectors if s in pivot_c.columns]]
pivot_c = pivot_c.loc[pivot_c.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 6))
pivot_c.plot.bar(stacked=True, ax=ax,
                 color=[SECTOR_COLOURS[s] for s in pivot_c.columns],
                 alpha=0.85, width=0.65)
_style_ax(ax, "Sector-level H2 demand — top 8 countries at 2050 (central)",
          "TWh / yr", xlabel="")
ax.legend(fontsize=8, frameon=False, loc="upper right", ncol=2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
fig.tight_layout()
fig.savefig(OUT_DIR / "country_sector_breakdown_central.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Country demand concentration (Lorenz curve) at 2050 ---
fig, ax = plt.subplots(figsize=(8, 6))

for bname, colour in SCENARIO_COLOURS.items():
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    totals = df_50.groupby("country")["h2_demand_mwh_per_yr"].sum().sort_values()
    cum = np.cumsum(totals.values) / totals.sum()
    x = np.arange(1, len(cum) + 1) / len(cum)
    ax.plot(x, cum, color=colour, lw=2, label=bname)

ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5, label="Perfect equality")
_style_ax(ax, "Demand concentration — Lorenz curve at 2050",
          "Cumulative share of EU-27 H2 demand", "Cumulative share of countries")
ax.legend(fontsize=9, frameon=False)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(OUT_DIR / "country_lorenz_2050.png", dpi=150, bbox_inches="tight")
plt.show()

# Gini coefficient
for bname in SCENARIO_COLOURS:
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    totals = df_50.groupby("country")["h2_demand_mwh_per_yr"].sum().sort_values().values
    n = len(totals)
    gini = (2 * np.sum((np.arange(1, n+1)) * totals) / (n * totals.sum())) - (n + 1) / n
    print(f"  {bname:20s} Gini = {gini:.3f}")

---
## 22 — Sector composition trajectories — all four bundles

Stacked area charts showing how the six sectors evolve from 2019 to 2050
in each scenario, revealing structural differences in the demand mix.

In [ ]:
# --- 4-panel stacked area: sector composition per bundle ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

for ax, (bname, colour) in zip(axes.flatten(), SCENARIO_COLOURS.items()):
    df_b = all_scenarios[bname]
    pivot_b = df_b.pivot_table(
        index="year", columns="sector",
        values="h2_demand_mwh_per_yr", aggfunc="sum", fill_value=0,
    ) / 1e6
    pivot_b = pivot_b[[s for s in ordered_sectors if s in pivot_b.columns]]

    ax.stackplot(
        pivot_b.index, *[pivot_b[s] for s in pivot_b.columns],
        labels=[s.capitalize() for s in pivot_b.columns],
        colors=[SECTOR_COLOURS[s] for s in pivot_b.columns],
        alpha=0.85,
    )
    _style_ax(ax, bname, "TWh / yr")
    ax.set_xlim(2019, 2050)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles[::-1], labels[::-1], loc="lower center", ncol=6,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("EU-27 H2 demand by sector — all scenarios (naphtha-constrained)",
             fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
fig.savefig(OUT_DIR / "sector_stacked_all_bundles.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 23 — Refinery-olefins coherence check

Direct comparison of refinery and olefins H2 demand trajectories to verify
that the naphtha supply constraint eliminates the previous inconsistency
where fossil olefins persisted after refinery shutdown.

In [ ]:
# --- Refinery vs olefins H2 demand side-by-side, all bundles ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (bname, colour) in zip(axes.flatten(), SCENARIO_COLOURS.items()):
    df_b = all_scenarios[bname]

    for sec, sec_colour, ls in [("refinery", SECTOR_COLOURS["refinery"], "-"),
                                  ("olefins", SECTOR_COLOURS["olefins"], "-")]:
        df_sec = df_b[df_b["sector"] == sec]
        agg = df_sec.groupby("year")["h2_demand_mwh_per_yr"].sum() / 1e6
        ax.plot(agg.index, agg.values, color=sec_colour, lw=2, ls=ls,
                label=sec.capitalize())

    _style_ax(ax, bname, "TWh / yr")
    ax.set_xlim(2019, 2050)
    ax.legend(fontsize=9, frameon=False)

    for sec, sec_colour in [("refinery", SECTOR_COLOURS["refinery"]),
                              ("olefins", SECTOR_COLOURS["olefins"])]:
        df_sec = df_b[df_b["sector"] == sec]
        val_50 = df_sec[df_sec["year"] == 2050]["h2_demand_mwh_per_yr"].sum() / 1e6
        ax.annotate(f"{val_50:.0f} TWh", xy=(2050, val_50),
                    fontsize=7, color=sec_colour, fontweight="bold",
                    textcoords="offset points", xytext=(-50, 5))

fig.suptitle("Refinery vs Olefins H2 demand (naphtha-constrained)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "refinery_vs_olefins_coherence.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 24 — Country x scenario heatmap at 2050

Which countries see the biggest demand shifts across scenarios?

In [ ]:
# --- Heatmap: country x scenario total demand at 2050 ---
heat_data = {}
for bname in SCENARIO_COLOURS:
    df_50 = all_scenarios[bname][all_scenarios[bname]["year"] == 2050]
    heat_data[bname] = df_50.groupby("country")["h2_demand_mwh_per_yr"].sum() / 1e6

heat_df = pd.DataFrame(heat_data)
heat_df = heat_df.loc[heat_df.max(axis=1).nlargest(15).index]
heat_df = heat_df.sort_values("central", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat_df.values, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, fontsize=9)
ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=9)

for i in range(len(heat_df.index)):
    for j in range(len(heat_df.columns)):
        val = heat_df.iloc[i, j]
        colour = "white" if val > heat_df.values.max() * 0.6 else "black"
        ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7, color=colour)

ax.set_title("Total H2 demand at 2050 (TWh) — top 15 countries",
             fontsize=11, fontweight="bold", pad=10)
fig.colorbar(im, ax=ax, label="TWh / yr", shrink=0.8)
fig.tight_layout()
fig.savefig(OUT_DIR / "country_scenario_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 25 — Demand uncertainty by country

For each country, the bar shows the range between low_h2 and
industry_stress at 2050 — a measure of scenario sensitivity.

In [ ]:
# --- Scenario spread per country at 2050 ---
df_low  = all_scenarios["low_h2"][all_scenarios["low_h2"]["year"] == 2050]
df_high = all_scenarios["industry_stress"][all_scenarios["industry_stress"]["year"] == 2050]

low_by_cc  = df_low.groupby("country")["h2_demand_mwh_per_yr"].sum() / 1e6
high_by_cc = df_high.groupby("country")["h2_demand_mwh_per_yr"].sum() / 1e6
spread = (high_by_cc - low_by_cc).sort_values(ascending=True)
top15 = spread.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(top15))
ax.barh(y, low_by_cc[top15.index].values,
        color=SCENARIO_COLOURS["low_h2"], alpha=0.8, label="low_h2")
ax.barh(y, top15.values, left=low_by_cc[top15.index].values,
        color=SCENARIO_COLOURS["industry_stress"], alpha=0.5, label="Delta to stress")
ax.set_yticks(y)
ax.set_yticklabels(top15.index, fontsize=9)
_style_ax(ax, "H2 demand range at 2050 — low_h2 to industry_stress",
          "TWh / yr", xlabel="")
ax.set_xlabel("TWh / yr", fontsize=9)
ax.legend(fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "country_demand_spread.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

| Function | Purpose |
|---|---|
| `list_bundles()` | List available YAML scenario bundles |
| `get_bundle_params(name)` | Inspect raw parameters without running |
| `load_bundle(name, ...)` | Run a complete scenario (all 6 sectors) |
| `project_*_h2_demand()` | Per-sector projection functions |
| `build_olefins_pathway_mix()` | Olefins four-pathway share trajectories |
| `apply_naphtha_supply_constraint()` | Cap fossil share vs refinery naphtha |
| `compute_olefins_h2_demand()` | Per-pathway H2 demand from production |
| `get_naphtha_for_crackers()` | Extract naphtha supply from CONCAWE config |
| `apply_petrochem_naphtha_correction()` | Remove cracker-bound naphtha HT |
| `build_refinery_unit_allocation()` | CONCAWE unit-feed allocation |

Key features:
- **Four-pathway olefins model**: MTO, bio-naphtha, chem recycling, fossil with independent ramps
- **Endogenous naphtha balance**: Fossil olefins capped by refinery naphtha supply (CONCAWE coupling)
- **Petrochem naphtha correction**: Prevents double-counting between refinery and olefins modules
- **Nested YAML config**: Per-pathway share targets and ramp timings in `scenario_registry.yaml`

In [ ]:
print("\u2713 How-To-Use notebook complete.")
print(f"  Generated figures in: {OUT_DIR}")